# 🔍 The Language Detective — How Machines Read Text
**Course:** Deep Learning Lab | **Notebook:** 03  
**Instructions:** Run every cell top to bottom. Fill in the 🔬 Observation and 📝 Reflection markdown cells. Do not modify any code cell.

In [ ]:
!pip install transformers tokenizers datasets spacy gensim sentence-transformers -q
!pip install tiktoken langdetect nltk scikit-learn matplotlib seaborn umap-learn rank-bm25 -q
!python -m spacy download en_core_web_sm -q
import sys
print("Install complete. Python:", sys.version)

In [ ]:
import nltk
for pkg in ['movie_reviews','stopwords','punkt','wordnet','averaged_perceptron_tagger','punkt_tab']:
    nltk.download(pkg, quiet=True)
print("NLTK ready.")
import torch, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import re, unicodedata, time, warnings, os, urllib.request, json
from collections import Counter, defaultdict
import pandas as pd
warnings.filterwarnings('ignore')
torch.manual_seed(42); np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"All imports done. Device: {device}")

---
# 📝 BLOCK 1 — TEXT IS DATA
*Before a model can learn from text, text must become numbers. This block shows every step of that transformation.*

In [ ]:
# Demo 1: String representations
print("Step 1: 'Hello \u4e16\u754c' at every level of representation...")
s = 'Hello \u4e16\u754c'
utf8  = s.encode('utf-8')
cpts  = [ord(c) for c in s]
print(f"Python string:      {repr(s)}")
print(f"UTF-8 bytes:        {utf8}")
print(f"Hex:                {utf8.hex()}")
print(f"Unicode codepoints: {cpts}")
print(f"\nThe string has {len(s)} characters but {len(utf8)} bytes in UTF-8.")
print(f"ASCII chars (Hello+space): 1 byte each = 6 bytes")
print(f"CJK chars (\u4e16\u754c): 3 bytes each = 6 bytes")
print(f"Total: 6 + 6 = {6+6} bytes \u2713")

In [ ]:
# Demo 2: Mojibake — wrong encoding
print("Step 2: Mojibake \u2014 decoding UTF-8 bytes as Latin-1...")
original = "Caf\u00e9 r\u00e9sum\u00e9 na\u00efve"
utf8_bytes = original.encode('utf-8')
garbled = utf8_bytes.decode('latin-1')
print(f"Original:  '{original}'")
print(f"Garbled:   '{garbled}'")
print(f"\nBytes for '\u00e9' (UTF-8): {chr(233).encode('utf-8').hex()} = 2 bytes")
print(f"Latin-1 reads each byte separately \u2192 two replacement chars")
print(f"RULE: Always declare encoding. UTF-8 is the safe default.")
print(f"This happens silently in production \u2014 files look fine until a character triggers it.")

In [ ]:
# Demo 3: Regex — emails, dates, HTML stripping
print("Step 3: Regex text extraction...")
blob = """
Email: support@example.com and admin@corp.org for help.
Dates: 2024-01-15, 03/28/2024, and December 5th 2023.
<p>Please <b>click here</b> to <a href='#'>visit us</a>.</p>
"""
emails = re.findall(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b', blob)
dates  = re.findall(r'\b(\d{4}-\d{2}-\d{2}|\d{2}/\d{2}/\d{4})\b', blob)
clean  = re.sub(r'<[^>]+>', '', blob).strip()
print(f"Emails found ({len(emails)}): {emails}")
print(f"Dates found  ({len(dates)}):  {dates}")
print(f"\nBefore HTML strip:\n{blob.strip()}")
print(f"\nAfter HTML strip:\n{clean}")

In [ ]:
# Demo 4: Text normalisation pipeline
print("Step 4: Normalisation pipeline on 'Don't you LOVE NLP?! R\u00e9sum\u00e9 here'...")
text = "Don't you LOVE NLP?! R\u00e9sum\u00e9 here"
contractions = {"don't":"do not","Don't":"do not","won't":"will not","can't":"cannot","I'm":"I am","it's":"it is"}

step1 = text.lower()
print(f"Original: {text}")
print(f"Step 1 (lowercase):           {step1}")

step2 = unicodedata.normalize('NFC', step1)
print(f"Step 2 (NFC normalisation):   {step2}")

step3 = step2
for k,v in contractions.items():
    step3 = step3.replace(k.lower(), v)
print(f"Step 3 (expand contractions): {step3}")

step4 = re.sub(r'[^\w\s]','',step3)
print(f"Step 4 (remove punctuation):  {step4}")

print("\nNFC: '\u00e9' stored as 1 codepoint. NFD stores it as e + combining accent = 2 codepoints.")
print("Without NFC, 'caf\u00e9' == 'caf\u00e9' returns False even though they look identical!")

In [ ]:
# Demo 5+6: Stop word removal and Stemming vs Lemmatisation
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

stop_words = set(stopwords.words('english'))
stemmer    = PorterStemmer()
lemma      = WordNetLemmatizer()

# Stop words
text_sw = "This is not a good product"
tokens  = text_sw.split()
filtered = [w for w in tokens if w.lower() not in stop_words]
print(f"Stop word removal:")
print(f"  Original:  '{text_sw}'")
print(f"  Filtered:  {filtered}")
print(f"  WARNING: 'not' was removed. Sentiment has been DESTROYED.")
print(f"  Remaining tokens imply: positive review. Original is: NEGATIVE.\n")

# Stemming vs Lemmatisation
words = ["running","better","mice","studies","argue","argument","feet","worse"]
print(f"{'Word':<15} {'Stem':<20} {'Lemma':<15} {'Lemma=real word?'}")
print("-"*60)
real = {"run","good","mouse","study","argue","argument","foot","bad","better","worse"}
for w in words:
    s=stemmer.stem(w); l=lemma.lemmatize(w)
    is_real = "\u2713" if l in real or l==w else "?"
    print(f"{w:<15} {s:<20} {l:<15} {is_real}")
print("\nStemming: fast, rule-based, often non-words ('studi','argu')")
print("Lemmatisation: dictionary-based, always real words. Prefer for ML tasks.")

In [ ]:
# Demo 7+8: spaCy NER and language detection
import spacy
nlp = spacy.load("en_core_web_sm")
from langdetect import detect, detect_langs

news = ("Apple Inc. CEO Tim Cook announced Tuesday that the company will invest "
        "$500 million in Texas. Microsoft founder Bill Gates praised the move from Seattle.")
doc  = nlp(news)
print("Named Entity Recognition:")
print(f"{'Entity':<30} {'Label':<12} Description")
print("-"*60)
for ent in doc.ents:
    print(f"{ent.text:<30} {ent.label_:<12} {spacy.explain(ent.label_) or ''}")

print(f"\nTotal entities: {len(doc.ents)}\n")

sentences = [
    ("Hello, how are you today?","English"),
    ("Bonjour, comment allez-vous?","French"),
    ("Hola, \u00bfc\u00f3mo est\u00e1s?","Spanish"),
    ("Guten Morgen, wie geht es?","German"),
    ("\u3053\u3093\u306b\u3061\u306f\u3001\u304a\u5143\u6c17\u3067\u3059\u304b\uff1f","Japanese"),
]
print("Language detection:")
print(f"{'Sentence':<40} {'Expected':>10} {'Detected':>10}")
print("-"*62)
for text,expected in sentences:
    detected = detect(text)
    match = "\u2713" if expected[:2].lower() in detected or detected in expected.lower() else "?"
    print(f"{text[:38]:<40} {expected:>10} {detected:>10} {match}")

### 🔬 Your Observation — Block 1
*In 3–5 sentences: why does removing 'not' destroy sentiment? What is mojibake and in what real scenario does it cause problems? Why does NFC normalisation matter for text equality checks?*

*(Write your answer here.)*

### 📝 Block Reflection — Block 1: Text is Data
*In 5–7 sentences, summarise the preprocessing decisions. Pick one real-world text pipeline (customer reviews, medical notes, tweets) and describe which steps you'd include or skip, and why.*

*(Write your answer here.)*

---
# ✂️ BLOCK 2 — TOKENIZATION
*Tokenization converts raw text into integer IDs that neural networks process. The tokenizer choice profoundly affects what models can learn.*

In [ ]:
# Demo 1: BPE from scratch
print("Step 1: Byte-Pair Encoding from scratch on toy corpus...")

corpus = ["low low low", "lower lower", "lowest"]
print(f"Corpus: {corpus}\n")

def get_vocab(corpus):
    vocab = {}
    for sentence in corpus:
        for word in sentence.split():
            key = " ".join(list(word)) + " </w>"
            vocab[key] = vocab.get(key,0) + sentence.split().count(word)
    return vocab

def get_pairs(vocab):
    pairs = defaultdict(int)
    for word,freq in vocab.items():
        syms = word.split()
        for i in range(len(syms)-1):
            pairs[(syms[i],syms[i+1])] += freq
    return pairs

def merge_vocab(pair, vocab):
    new_vocab = {}
    bigram = re.escape(" ".join(pair))
    pattern = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in vocab:
        new_vocab[pattern.sub("".join(pair), word)] = vocab[word]
    return new_vocab

vocab = get_vocab(corpus)
print("Initial character vocabulary:")
for w,f in vocab.items(): print(f"  {w!r}: {f}")

print("\nBPE merge steps:")
for step in range(5):
    pairs = get_pairs(vocab)
    if not pairs: break
    best  = max(pairs, key=pairs.get)
    vocab = merge_vocab(best, vocab)
    v_size = len(set(s for w in vocab for s in w.split()))
    print(f"\nMerge {step+1}: '{best[0]}' + '{best[1]}' \u2192 '{best[0]+best[1]}'")
    for w,f in vocab.items(): print(f"  {w!r}: {f}")
    print(f"  Vocabulary size now: {v_size} tokens")

print("\nThis is exactly how GPT-2 and GPT-4 tokenizers are trained on large corpora.")

In [ ]:
# Demo 2+3: tiktoken and cost calculator
print("Step 2: tiktoken \u2014 GPT-4 tokenizer...")
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")

words = ["Hello, world!", "unbelievable", "eicosapentaenoic", "supercalifragilistic"]
print(f"{'Text':<30} {'Token IDs':<35} {'N tokens'}")
print("-"*75)
for w in words:
    ids = enc.encode(w)
    decoded = [enc.decode([i]) for i in ids]
    print(f"{w:<30} {str(ids):<35} {len(ids)}")
    print(f"  {'Tokens:':<28} {decoded}")

paragraph = """Deep learning has revolutionised NLP over the past decade. Models like GPT-4
can generate coherent text, answer questions, and write code. The key innovation was
the attention mechanism, which allows models to weigh the relevance of different parts
of the input when generating each output token. Training requires massive datasets."""
tokens = enc.encode(paragraph)
n_tok  = len(tokens)
cost   = n_tok / 1_000_000 * 5.0
print(f"\n--- Cost Calculator ---")
print(f"Paragraph: ~{len(paragraph.split())} words  |  {n_tok} tokens")
print(f"At GPT-4o pricing ($5.00/1M tokens): ${cost:.6f} per paragraph")
print(f"To process 1,000,000 paragraphs: ${cost*1_000_000:.2f}")

In [ ]:
# Demo 4: Language tax
print("Step 3: Language tax \u2014 token counts across 8 languages...")
translations = {
    "English":  "Hello, how are you?",
    "French":   "Bonjour, comment allez-vous?",
    "German":   "Hallo, wie geht es Ihnen?",
    "Spanish":  "Hola, \u00bfc\u00f3mo est\u00e1s?",
    "Russian":  "\u041f\u0440\u0438\u0432\u0435\u0442, \u043a\u0430\u043a \u0442\u044b?",
    "Arabic":   "\u0645\u0631\u062d\u0628\u0627\u060c \u0643\u064a\u0641 \u062d\u0627\u0644\u0643\u061f",
    "Chinese":  "\u4f60\u597d\uff0c\u4f60\u597d\u5417\uff1f",
    "Japanese": "\u3053\u3093\u306b\u3061\u306f\u3001\u304a\u5143\u6c17\u3067\u3059\u304b\uff1f",
}
enc2 = tiktoken.get_encoding("cl100k_base")
results = {}
for lang,text in translations.items():
    n = len(enc2.encode(text))
    results[lang] = n
    print(f"  {lang:<12}: {n:3d} tokens  '{text}'")

en_n = results["English"]
print("\nOverhead vs English:")
for lang,n in sorted(results.items(),key=lambda x:x[1]):
    print(f"  {lang:<12}: {n/en_n:.2f}x  {'\u2588'*int(n/en_n*10)}")

fig,ax=plt.subplots(figsize=(10,5))
langs,counts=zip(*sorted(results.items(),key=lambda x:x[1]))
cols=['tomato' if c==max(counts) else 'steelblue' for c in counts]
bars=ax.bar(langs,counts,color=cols)
ax.axhline(en_n,color='gray',ls='--',label='English baseline')
for bar,c in zip(bars,counts): ax.text(bar.get_x()+bar.get_width()/2,c+0.05,str(c),ha='center',fontsize=9)
ax.set_ylabel("Token count"); ax.set_title("Figure 2-1: Token count per language for equivalent phrases")
ax.legend(); plt.tight_layout(); plt.show()
max_lang=max(results,key=lambda k:results[k])
print(f"\n{max_lang} uses {results[max_lang]/en_n:.1f}x more tokens than English for equivalent meaning.")
print("This 'language tax' means shorter effective context windows for non-English users.")
print("Caption: GPT-4 tokenizer was trained primarily on English \u2014 non-English languages pay an efficiency penalty.")

In [ ]:
# Demo 5+6: BERT WordPiece and wrong tokenizer demo
print("Step 4: BERT WordPiece tokenization...")
from transformers import BertTokenizer, GPT2Tokenizer

bert_tok = BertTokenizer.from_pretrained("bert-base-uncased")
words_wp  = ["playing","unbelievably","tokenization","supercalifragilistic","transformers"]
print(f"{'Word':<25} {'Tokens':<40} {'IDs'}")
print("-"*80)
for w in words_wp:
    toks = bert_tok.tokenize(w)
    ids  = bert_tok.convert_tokens_to_ids(toks)
    print(f"{w:<25} {str(toks):<40} {ids}")

print("\n## prefix = subword continuation token (not a word boundary).")
print("Example: 'unbelievably' \u2192 ['un','##bel','##iev','##ably']")
print("This allows a 30,000-word vocabulary to represent almost any English word.\n")

# Wrong tokenizer
gpt2_tok  = GPT2Tokenizer.from_pretrained("gpt2")
text_demo = "The cat sat on the mat"
bert_ids  = bert_tok.encode(text_demo, add_special_tokens=False)
gpt2_ids  = gpt2_tok.encode(text_demo)
print(f"Step 5: Wrong tokenizer demo \u2014 text: '{text_demo}'")
print(f"\nBERT tokenizer correctly:   {bert_tok.tokenize(text_demo)}")
print(f"BERT token IDs:             {bert_ids}")
print(f"\nGPT-2 tokenizer correctly:  {[gpt2_tok.decode([i]) for i in gpt2_ids]}")
print(f"GPT-2 token IDs:            {gpt2_ids}")
print(f"\nIf you feed BERT IDs to GPT-2 embedding table:")
for bid in bert_ids[:4]:
    gpt2_word = repr(gpt2_tok.decode([bid]))
    bert_word = bert_tok.decode([bid])
    print(f"  ID {bid:5d} \u2192 BERT='{bert_word}'  GPT-2={gpt2_word}  \u2190 COMPLETELY DIFFERENT WORD")
print("\nRULE: tokenizer and model must ALWAYS come from the same family.")
print("This bug doesn't crash \u2014 it silently corrupts every single input.")

### 🔬 Your Observation — Block 2
*In 3–5 sentences: why does the language tax matter for global AI product fairness? What does the ## prefix mean in BERT tokens? Why is using the wrong tokenizer a 'silent failure' (doesn't raise an error)?*

*(Write your answer here.)*

### 📝 Block Reflection — Block 2: Tokenization
*In 5–7 sentences, summarise key insights. Describe a scenario where a poorly chosen tokenizer would hurt performance on a specific domain (medical text, code, tweets).*

*(Write your answer here.)*

---
# 🧠 BLOCK 3 — REPRESENTATIONS
*How do tokens become vectors that capture meaning? This block traces the evolution from one-hot to dense embeddings.*

In [ ]:
# Demo 1+2+3: One-hot, BoW, TF-IDF
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# One-hot
vocab = ["cat","dog","bulldozer","fish","car"]
w2i   = {w:i for i,w in enumerate(vocab)}
V     = len(vocab)
def oh(w): v=np.zeros(V); v[w2i[w]]=1; return v
def cos(a,b): return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)+1e-10)

print("One-hot cosine similarities:")
print(f"  sim(cat, dog)       = {cos(oh('cat'),oh('dog')):.4f}")
print(f"  sim(cat, bulldozer) = {cos(oh('cat'),oh('bulldozer')):.4f}")
print(f"  sim(dog, bulldozer) = {cos(oh('dog'),oh('bulldozer')):.4f}")
print("All distances = 0.00. The model cannot tell cat and dog are more related than cat and bulldozer.\n")

# BoW
doc1="dog bites man"; doc2="man bites dog"
all_w=sorted(set(doc1.split()+doc2.split()))
def bow(d,vocab): c=Counter(d.split()); return np.array([c.get(w,0) for w in vocab])
bv1=bow(doc1,all_w); bv2=bow(doc2,all_w)
print(f"BoW: '{doc1}' = {bv1}  |  '{doc2}' = {bv2}")
print(f"BoW cosine similarity = {cos(bv1,bv2):.4f}")
print("BoW similarity = 1.00. Word order is COMPLETELY INVISIBLE.\n")

# TF-IDF
docs=["the cat sat on the mat","the dog ate the cat food","machine learning is fun",
      "deep learning models learn patterns","the cat and the dog are friends"]
tfidf=TfidfVectorizer(); X=tfidf.fit_transform(docs)
vocab_t=tfidf.get_feature_names_out()
df=pd.DataFrame(X.toarray().round(3),columns=vocab_t,index=[f"d{i+1}" for i in range(5)])
zero_w=[w for w in vocab_t if df[w].max()<0.01]
best_w=df.max(axis=0).idxmax()
print(f"TF-IDF matrix (top rows, selected cols):\n{df[list(vocab_t[:8])].to_string()}\n")
print(f"Words with near-zero TF-IDF (appear in all docs): {zero_w}")
print(f"Highest TF-IDF word: '{best_w}' \u2014 appears in few docs, very distinctive.")

In [ ]:
# Demo 4+5: GloVe word vectors
print("Step 4: GloVe word embeddings and the king-man+woman analogy...")

glove_file = "glove.6B.50d.txt"
if not os.path.exists(glove_file):
    print("  Downloading GloVe 50d (~170MB)...")
    try:
        import zipfile
        urllib.request.urlretrieve("https://nlp.stanford.edu/data/glove.6B.zip","glove.zip")
        with zipfile.ZipFile("glove.zip","r") as z: z.extract("glove.6B.50d.txt",".")
        print("  GloVe downloaded.")
    except Exception as e:
        print(f"  Download failed: {e}"); glove_file=None

if glove_file and os.path.exists(glove_file):
    glove={}
    with open(glove_file,'r',encoding='utf-8') as f:
        for line in f:
            p=line.split()
            if len(p)==51: glove[p[0]]=np.array(p[1:],dtype=np.float32)
    print(f"  Loaded {len(glove):,} vectors (dim=50)")

    def nearest(vec,glove,exclude=[],n=5):
        sims={w:np.dot(vec,v)/(np.linalg.norm(vec)*np.linalg.norm(v)+1e-10)
              for w,v in glove.items() if w not in exclude}
        return sorted(sims.items(),key=lambda x:-x[1])[:n]

    if all(w in glove for w in ['king','man','woman']):
        av=glove['king']-glove['man']+glove['woman']
        res=nearest(av,glove,exclude=['king','man','woman'])
        print(f"\nking \u2212 man + woman = ?")
        for i,(w,s) in enumerate(res): print(f"  #{i+1}: {w:<15} cosine={s:.4f}")
        print(f"'queen' in top 3: {any(w=='queen' for w,_ in res[:3])}")

    if all(w in glove for w in ['doctor','man','woman']):
        bv=glove['doctor']-glove['man']+glove['woman']
        br=nearest(bv,glove,exclude=['doctor','man','woman'],n=3)
        print(f"\nBias audit: doctor \u2212 man + woman = ?")
        for w,s in br: print(f"  {w:<15} {s:.4f}")
        print("  This reflects gender bias in the training corpus.")
        print("  If the result is a female-stereotyped profession, the corpus encoded that association.")
else:
    print("GloVe not available. Key results from the original paper:")
    print("  king \u2212 man + woman \u2192 queen (cosine \u2248 0.85)")
    print("  paris \u2212 france + germany \u2192 berlin (cosine \u2248 0.82)")
    print("  doctor \u2212 man + woman \u2192 nurse (corpus gender bias)")
    print("GloVe vectors encode semantic relationships as arithmetic in vector space.")
print("Caption: Word vector arithmetic reveals both linguistic structure and training corpus biases.")

In [ ]:
# Demo 6: Sentence-BERT
print("Step 5: Sentence-BERT \u2014 semantic sentence similarity...")
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

sbert = SentenceTransformer('all-MiniLM-L6-v2')
sentences = [
    "The cat sat on the mat.",
    "A feline rested on a rug.",
    "Machine learning is fascinating.",
    "I love deep learning and AI.",
    "The stock market fell sharply today.",
]
embs = sbert.encode(sentences)
sim  = cosine_similarity(embs)

print(f"Embedding shape: {embs.shape}  (5 sentences \u00d7 384 dims)")
print("\nCosine similarity matrix:")
for i,s in enumerate(sentences):
    row = "  ".join([f"{sim[i,j]:.2f}" for j in range(5)])
    print(f"  S{i+1}: {s[:35]:<36} | {row}")

np.fill_diagonal(sim,-1)
i_top,j_top=np.unravel_index(sim.argmax(),sim.shape)
print(f"\nMost similar: S{i_top+1} & S{j_top+1}  (sim={sim[i_top,j_top]:.3f})")
print(f"  '{sentences[i_top]}'")
print(f"  '{sentences[j_top]}'")
print("These share almost no words yet are semantically equivalent \u2014 SBERT captures meaning, not surface form.")

fig,ax=plt.subplots(figsize=(8,6))
np.fill_diagonal(sim,1.0)
short=[f"S{i+1}: {s[:22]}..." for i,s in enumerate(sentences)]
sns.heatmap(sim,annot=True,fmt='.2f',cmap='RdYlGn',ax=ax,
            xticklabels=[f'S{i+1}' for i in range(5)],yticklabels=short,vmin=0,vmax=1)
ax.set_title("Figure 3-1: Sentence-BERT semantic similarity matrix")
plt.tight_layout(); plt.show()
print("Caption: S1 ('cat on mat') and S2 ('feline on rug') score high despite different words.")

### 🔬 Your Observation — Block 3
*In 3–5 sentences: why can't one-hot vectors detect that 'cat' and 'dog' are related? What does the king−man+woman analogy reveal about how meaning is encoded? Did Sentence-BERT correctly find the most semantically similar pair?*

*(Write your answer here.)*

### 📝 Block Reflection — Block 3: Representations
*In 5–7 sentences, trace the evolution from one-hot \u2192 GloVe \u2192 Sentence-BERT. Explain why each step is an improvement. Describe one application where SBERT would dramatically outperform TF-IDF.*

*(Write your answer here.)*

---
# 🤖 BLOCK 4 — CLASSICAL MODELS
*Before transformers dominated NLP, classical methods were state of the art. Understanding them reveals what transformers actually improved.*

In [ ]:
# Demo 1: Naive Bayes spam classifier
print("Step 1: Naive Bayes spam classifier...")
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Download SMS spam corpus
sms_url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
try:
    urllib.request.urlretrieve(sms_url,"sms.tsv")
    with open("sms.tsv","r",encoding="utf-8") as f: lines=f.readlines()
    labels=[l.split('\t')[0] for l in lines]; texts=[l.split('\t')[1].strip() for l in lines]
    pos_class='spam'
    print(f"SMS corpus: {len(texts)} messages, {labels.count('spam')} spam, {labels.count('ham')} ham")
except Exception as e:
    print(f"SMS download failed ({e}), using movie reviews...")
    from nltk.corpus import movie_reviews
    texts=[" ".join(movie_reviews.words(f)) for f in movie_reviews.fileids()]
    labels=[movie_reviews.categories(f)[0] for f in movie_reviews.fileids()]
    pos_class='pos'
    print(f"Movie reviews: {len(texts)} docs, classes: {set(labels)}")

t0=time.time()
vec=CountVectorizer(stop_words='english',max_features=5000)
X=vec.fit_transform(texts); y=np.array(labels)
X_tr,X_te,y_tr,y_te=train_test_split(X,y,test_size=0.2,random_state=42)
clf=MultinomialNB(); clf.fit(X_tr,y_tr)
train_ms=(time.time()-t0)*1000
y_pred=clf.predict(X_te)

acc=accuracy_score(y_te,y_pred)
prec=precision_score(y_te,y_pred,pos_label=pos_class,average='binary',zero_division=0)
rec=recall_score(y_te,y_pred,pos_label=pos_class,average='binary',zero_division=0)
f1=f1_score(y_te,y_pred,pos_label=pos_class,average='binary',zero_division=0)
print(f"\nNaive Bayes: accuracy={acc:.4f}  precision={prec:.4f}  recall={rec:.4f}  F1={f1:.4f}")
print(f"Trained in {train_ms:.0f}ms. This is the baseline every team should run first.")

customs=["You won a FREE lottery ticket! Claim now!","Hi, are you coming to the meeting?","URGENT: Verify your account NOW","See you at 3pm"]
print("\nCustom predictions:")
for s in customs:
    p=clf.predict(vec.transform([s]))[0]; print(f"  [{p.upper()}] {s}")

In [ ]:
# Demo 2: Vanishing gradient in vanilla RNN
print("Step 2: Vanishing gradient in a vanilla RNN...")
import torch.nn as nn

class TinyRNN(nn.Module):
    def __init__(self,in_sz=10,hid_sz=20):
        super().__init__()
        self.Wx=nn.Linear(in_sz,hid_sz,bias=False)
        self.Wh=nn.Linear(hid_sz,hid_sz,bias=True)
    def forward(self,x,h): return torch.tanh(self.Wx(x)+self.Wh(h))

T=20; rnn=TinyRNN()
inputs=[torch.randn(1,10) for _ in range(T)]
for inp in inputs: inp.requires_grad_(True)
h=torch.zeros(1,20)
for t,x in enumerate(inputs): h=rnn(x,h)
h.sum().backward()

grad_norms=[]
for t,inp in enumerate(inputs):
    g=inp.grad.norm().item() if inp.grad is not None else 0.0
    grad_norms.append(g)

print(f"Gradient norms (step 0 = earliest input):")
for t,g in enumerate(grad_norms):
    bar='\u2588'*max(1,int(g/max(grad_norms+[1e-9])*20))
    print(f"  step {t:2d}: {g:.2e}  {bar}")
print(f"\nAt step 19 (most recent): {grad_norms[-1]:.6f}")
print(f"At step  0 (earliest):   {grad_norms[0]:.6f}")
if grad_norms[0]>0:
    print(f"Ratio: {grad_norms[-1]/grad_norms[0]:.1f}x larger gradient at step 19 vs step 0")
print("\nThis is the vanishing gradient problem.")
print("Early tokens receive almost no learning signal \u2192 model forgets long-range dependencies.")

fig,ax=plt.subplots(figsize=(10,4))
ax.bar(range(T),grad_norms,color=['red' if g<1e-4 else 'steelblue' for g in grad_norms])
ax.set_xlabel("Time step"); ax.set_ylabel("Gradient magnitude")
ax.set_title("Figure 4-1: Vanishing gradient \u2014 gradient shrinks as it travels back through time")
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
print("Caption: Red bars = near-zero gradient = the model barely learns from early tokens.")

In [ ]:
print("Step 3: Attention weights in an English→French translation model...")
from transformers import MarianMTModel, MarianTokenizer

model_name="Helsinki-NLP/opus-mt-en-fr"
print(f"Loading {model_name}...")

# Try to load model to GPU, fallback to CPU if CUDA out of memory
try:
    tok_mt=MarianTokenizer.from_pretrained(model_name)
    model_mt=MarianMTModel.from_pretrained(model_name).to(device)
    model_mt.eval()
except RuntimeError as e:
    if "CUDA out of memory" in str(e):
        print("CUDA out of memory. Falling back to CPU for model loading.")
        # Use a local device variable for this model only, to not affect other parts if `device` is globally used
        current_model_device = torch.device('cpu')
        tok_mt=MarianTokenizer.from_pretrained(model_name)
        model_mt=MarianMTModel.from_pretrained(model_name).to(current_model_device)
        model_mt.eval()
        # Ensure subsequent tensor operations for this model also use the CPU
        device = current_model_device # Update global device to CPU for subsequent steps in this cell
    else:
        raise e # Re-raise other runtime errors

src="The cat sat on the mat"
inp=tok_mt(src,return_tensors="pt").to(device)
with torch.no_grad():
    out=model_mt.generate(**inp,output_attentions=True,
                           return_dict_in_generate=True,max_new_tokens=30)
decoded=tok_mt.decode(out.sequences[0],skip_special_tokens=True)
print(f"Source (EN):      '{src}'")
print(f"Translation (FR): '{decoded}'")

src_toks=tok_mt.tokenize(src)+['</s>']
tgt_toks=tok_mt.convert_ids_to_tokens(out.sequences[0])
print(f"\nSource tokens: {src_toks}")
print(f"Target tokens: {tgt_toks}")

if hasattr(out,'cross_attentions') and out.cross_attentions:
    try:
        # Get the specific attention tensor (e.g., first element for batch=0 from the last layer)
        attention_tensor = out.cross_attentions[-1][0]
        if attention_tensor is not None:
            ca=attention_tensor.mean(dim=0).cpu().numpy()
            n_tgt,n_src=min(ca.shape[0],len(tgt_toks)),min(ca.shape[1],len(src_toks))
            attn=ca[:n_tgt,:n_src]
            fig,ax=plt.subplots(figsize=(10,6))
            sns.heatmap(attn,ax=ax,xticklabels=src_toks[:n_src],yticklabels=tgt_toks[:n_tgt],
                        cmap='Blues',linewidths=0.3)
            ax.set_xlabel("Source tokens (EN)"); ax.set_ylabel("Target tokens (FR)")
            ax.set_title("Figure 4-2: Attention weights — which source token did the decoder look at?")
            plt.tight_layout(); plt.show()
            mi=np.unravel_index(attn.argmax(),attn.shape)
            print(f"Decoder attended most to '{src_toks[mi[1]]}' when generating '{tgt_toks[mi[0]]}'")
        else:
            print("Attention: Could not visualize attention weights because the specific attention tensor was None.")
            print("Expected: 'chat' attends to 'cat', 'assis' attends to 'sat'.")
    except (IndexError, TypeError, AttributeError) as e:
        # Catch issues if cross_attentions structure is not as expected,
        # e.g., if indexing fails due to empty list/tuple or NoneType deeper within.
        print(f"Attention: Could not visualize attention weights due to an unexpected error accessing attention data: {e}")
        print("Expected: 'chat' attends to 'cat', 'assis' attends to 'sat'.")
else:
    print("Attention: bright cell = decoder looked strongly at that source word when generating target word.")
    print("Expected: 'chat' attends to 'cat', 'assis' attends to 'sat'.")
print("Caption: Attention replaced the fixed-length bottleneck of seq2seq — the decoder can look back at any source position.")

### 🔬 Your Observation — Block 4
*In 3–5 sentences: how does Naive Bayes train in milliseconds yet stay reasonably accurate? What does the vanishing gradient graph show about why RNNs struggle with long sentences? In the attention heatmap, does the EN\u2192FR alignment make linguistic sense?*

*(Write your answer here.)*

### 📝 Block Reflection — Block 4: Classical Models
*In 5–7 sentences, summarise what classical NLP models do well and where they fail. Explain why the vanishing gradient problem motivated both LSTMs and Transformers.*

*(Write your answer here.)*

---
# 📊 BLOCK 5 — HONEST EVALUATION
*A good metric is honest about what the model can and cannot do. This block reveals how evaluation can mislead \u2014 and how to avoid it.*

In [ ]:
# Demo 1: BLEU score from scratch
print("Step 1: BLEU score from scratch...")
import math

def ngrams(tokens,n): return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def bleu(cand_str,ref_str,max_n=4):
    cand=cand_str.lower().split(); ref=ref_str.lower().split()
    bp=1.0 if len(cand)>=len(ref) else math.exp(1-len(ref)/len(cand))
    precs=[]
    for n in range(1,max_n+1):
        cn=Counter(ngrams(cand,n)); rn=Counter(ngrams(ref,n))
        clipped={ng:min(cnt,rn[ng]) for ng,cnt in cn.items()}
        denom=sum(cn.values())
        p=sum(clipped.values())/denom if denom>0 else 0
        precs.append(p)
        print(f"  {n}-gram precision: {p:.4f}")
    lp=sum(math.log(max(p,1e-10)) for p in precs)/max_n
    return bp*math.exp(lp)

pairs=[
    ("The cat is on the mat",   "The cat sat on the mat"),
    ("A feline rests on a rug", "The cat sat on the mat"),
    ("The cat sat on the mat",  "The cat sat on the mat"),
]
print("BLEU evaluation on 3 candidate/reference pairs:\n")
scores=[]
for i,(c,r) in enumerate(pairs):
    print(f"Pair {i+1}:")
    print(f"  Candidate: '{c}'\n  Reference: '{r}'")
    s=bleu(c,r)
    scores.append(s)
    print(f"  BLEU = {s:.4f}\n")

print(f"Pair 2 (semantically correct paraphrase) BLEU = {scores[1]:.4f}")
print(f"Pair 3 (exact match)                     BLEU = {scores[2]:.4f}")
print("Paraphrase scores MUCH lower than exact match \u2014 BLEU is a surface-level metric.")
print("RULE: BLEU measures n-gram overlap, not meaning. Always use multiple metrics.")

In [ ]:
# Demo 2: Perplexity
print("Step 2: Perplexity with GPT-2...")
from transformers import GPT2Tokenizer, GPT2LMHeadModel

gpt2_tok2=GPT2Tokenizer.from_pretrained("gpt2")
gpt2_mod2=GPT2LMHeadModel.from_pretrained("gpt2").to(device)
gpt2_mod2.eval()

def perplexity(text,model,tok):
    inp=tok(text,return_tensors="pt").to(device)
    with torch.no_grad(): out=model(**inp,labels=inp["input_ids"])
    return torch.exp(out.loss).item()

sentences=[
    "The dog sat on the mat.",
    "The cat sat on the mat.",
    "I enjoy breakfast in the morning.",
    "Purple ideas sleep furiously.",
    "Sat the mat on dog.",
]
print(f"\n{'Perplexity':>12}   Sentence")
print("-"*65)
ppls=[]
for s in sentences:
    p=perplexity(s,gpt2_mod2,gpt2_tok2); ppls.append(p)
    tag="\u2190 very natural" if p<100 else ("\u2190 very unnatural" if p>1000 else "\u2190 somewhat odd")
    print(f"{p:12.1f}   '{s}' {tag}")

print(f"\nLowest  ppl: '{sentences[np.argmin(ppls)]}' = {min(ppls):.1f}")
print(f"Highest ppl: '{sentences[np.argmax(ppls)]}' = {max(ppls):.1f}")
print("\nLower perplexity = model assigns higher probability = more natural-sounding text.")
print("Perplexity intuition: 'on average, how many equally likely choices did the model face at each token?'")

In [ ]:
# Demo 3: Test set leakage
print("Step 3: Test set leakage demonstration...")
from sklearn.linear_model import LogisticRegression

np.random.seed(42)
n=min(len(texts),1200)
idx=np.random.choice(len(texts),n,replace=False)
X_all=vec.transform([texts[i] for i in idx]); y_all=np.array([labels[i] for i in idx])
split=int(n*0.8)
X_tr2,X_te2=X_all[:split],X_all[split:]
y_tr2,y_te2=y_all[:split],y_all[split:]

# LEAKED: train on ALL data, evaluate on test
lr_leak=LogisticRegression(max_iter=300,random_state=42)
lr_leak.fit(X_all,y_all)
acc_leak=accuracy_score(y_te2,lr_leak.predict(X_te2))

# CORRECT: train only on train split
lr_ok=LogisticRegression(max_iter=300,random_state=42)
lr_ok.fit(X_tr2,y_tr2)
acc_ok=accuracy_score(y_te2,lr_ok.predict(X_te2))

print(f"Train size: {split}  |  Test size: {n-split}")
print(f"\nLeaked  accuracy (train+test combined): {acc_leak:.4f}  \u2190 INVALID \u2014 model saw test during training")
print(f"Correct accuracy (train only):          {acc_ok:.4f}  \u2190 VALID")
print(f"\nLeakage inflated accuracy by {(acc_leak-acc_ok)*100:.2f} percentage points.")
print(f"RULE: The test set must NEVER be seen during training OR feature engineering.")
print(f"Even fitting a TF-IDF vocabulary on the full dataset before splitting is a form of leakage!")

fig,ax=plt.subplots(figsize=(6,4))
ax.bar(["Leaked (invalid)","Correct (valid)"],[acc_leak,acc_ok],color=['tomato','steelblue'])
ax.set_ylim(0,1); ax.set_ylabel("Accuracy"); ax.axhline(acc_ok,color='black',ls='--')
ax.set_title(f"Figure 5-1: Test set leakage inflates accuracy by {(acc_leak-acc_ok)*100:.1f}pp")
for i,(v,label) in enumerate(zip([acc_leak,acc_ok],["Leaked","Correct"])):
    ax.text(i,v+0.01,f"{v:.4f}",ha='center',fontsize=10)
plt.tight_layout(); plt.show()
print("Caption: Leakage makes the model appear better than it is. Real-world deployment will expose the true performance.")

### 🔬 Your Observation — Block 5
*In 3–5 sentences: why does the semantically correct paraphrase score low on BLEU? What does high perplexity on 'Purple ideas sleep furiously' tell you about what GPT-2 learned? Why is test set leakage so dangerous \u2014 and why is it easy to introduce by accident?*

*(Write your answer here.)*

### 📝 Block Reflection — Block 5: Honest Evaluation
*In 5–7 sentences, summarise the key evaluation pitfalls. Design an evaluation protocol for an NLP application of your choice \u2014 list which metrics you would use, which you would avoid, and why.*

*(Write your answer here.)*

---
## 🏆 Lab Report — Final Reflection

*Answer all 5 questions below in full sentences (minimum 2 sentences each):*

**1. What was the single most surprising result you observed in this lab?**

*(Write your answer here.)*

**2. Which concept was hardest to understand before seeing it run? Did running it help?**

*(Write your answer here.)*

**3. Pick any one demo \u2014 explain WHY it produced the output it did, in your own words.**

*(Write your answer here.)*

**4. What would break if you changed one specific parameter? Which one and why?**

*(Write your answer here.)*

**5. How would you apply one technique from this lab to a real problem you care about?**

*(Write your answer here.)*

---
*Notebook complete. Submit with all Observation and Reflection cells filled in.*